<a href="https://colab.research.google.com/github/junkyul/Agentics/blob/main/tutorials/amap_reduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# aMapReduce Framework

Agentics enable scalable execution of LLM workflows by implementing a MapReduce framework which enable the async use of LLM blended with regular python code.

In [1]:
! pip install agentics-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.0 MB/s eta 0:00

In [2]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



On your google drive creare a new folder called ColumbiaClass and move there the .env file from your agentic project and tutorials/data/dow_jones.csv .
If you have a google drive app on your desktop, you can do that from your shell

```bash
> mkdir MYDRIVE_LOCATION/ColumbiaClass
> cp -r tutorials/data/dow_jones.csv MYDRIVE_LOCATION/ColumbiaClass/
> cp .env MYDRIVE_LOCATION/ColumbiaClass/
```



In [3]:
## check if .env and dow_jones.csv are there
!ls -a /content/drive/MyDrive/ColumbiaClass/

dow_jones.csv  .env


Let us first define an aType to represent StockMarket Data for the DowJones index, and populate it with historical data

In [5]:
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/ColumbiaClass/.env')
from agentics import Agentics as AG
from typing import Optional
from pydantic import BaseModel

## Define the data model for stock market data


class StockMarketState(BaseModel):
    Date: Optional[str] = None
    Open: Optional[float] = None
    High: Optional[float] = None
    Low: Optional[float] = None
    Close: Optional[float]  = None
    Volume: Optional[int] = None
    Adj_Close: Optional[float] = None
    daily_range: Optional[float] = None
    News: Optional[str] = None
    Explanation_report: Optional[str] = None




## import the data
dj_data = AG.from_csv("/content/drive/MyDrive/ColumbiaClass/dow_jones.csv", atype=StockMarketState)

2025-09-03 19:29:52.038 | DEBUG    | agentics.core.agentics:from_csv:312 - Importing Agentics of type StockMarketState from CSV /content/drive/MyDrive/ColumbiaClass/dow_jones.csv


## Amap

Amap functions enable async execution of functions over all the states of an AG. Agentics supports 1:1 maps that maps all states of an AG into states of the same type.

In the following example we define a simple function to compute the daily_range of the stock and we pass that to an amap fuction which applies that to all states asyncronously

In [6]:
## Note that input and output are both StockMarketState objects
async def get_daily_variation_percentage(state: StockMarketState) -> StockMarketState:
    state.daily_range = (float(state.High) - float(state.Low)) / float(state.Low) * 100
    return state

## Apply the function to all states using amap
dj_data.batch_size = 100
dj_data = await dj_data.amap(get_daily_variation_percentage)

for state in dj_data[:3]:
    print(f"Date: {state.Date}, Daily Range: {state.daily_range}")

2025-09-03 19:30:18.430 | DEBUG    | agentics.core.agentics:amap:206 - Executing amap on function <function get_daily_variation_percentage at 0x7992c5befce0>
2025-09-03 19:30:18.435 | DEBUG    | agentics.core.agentics:amap:231 - 100 states processed. 3.472566604614258e-05 seconds average per state in the last chunk ...
2025-09-03 19:30:18.439 | DEBUG    | agentics.core.agentics:amap:231 - 200 states processed. 3.340005874633789e-05 seconds average per state in the last chunk ...
2025-09-03 19:30:18.442 | DEBUG    | agentics.core.agentics:amap:231 - 300 states processed. 2.0041465759277344e-05 seconds average per state in the last chunk ...
2025-09-03 19:30:18.446 | DEBUG    | agentics.core.agentics:amap:231 - 400 states processed. 2.1936893463134767e-05 seconds average per state in the last chunk ...
2025-09-03 19:30:18.449 | DEBUG    | agentics.core.agentics:amap:231 - 500 states processed. 2.1109580993652345e-05 seconds average per state in the last chunk ...
2025-09-03 19:30:18.453 

Date: 2016-07-01, Daily Range: 0.47703930117312493
Date: 2016-06-30, Daily Range: 1.2353831025172834
Date: 2016-06-29, Daily Range: 1.423521751672572


## aReduce

Reduce functions enable executing operations on the entire list of elements (states) within an Agentics group. Although reduce operations are intrinsically synchronous—since they consider all states at once—they are defined as async functions to allow for internal async calls (such as fetching news or running LLMs).

In the following example we will use a reduce function to analyze get the top 10 days with highest variation in the market

In [7]:
async def get_highest_volatility_days(states:list[StockMarketState]) -> list[StockMarketState]:

    # sort the states by volatility and return the top 10, define a new AG with these states
    return sorted(states,
                key=lambda x: abs(x.daily_range) if x.daily_range is not None else 0,
                reverse=True)[:10]

# apply the reduce function to get the top 10 days with highest volatility
highest_volatility_days = await dj_data.areduce(get_highest_volatility_days)
print(highest_volatility_days.pretty_print())

Atype : <class '__main__.StockMarketState'>
Date: '2008-10-10'
Open: 8568.669922
High: 8901.280273
Low: 7882.509766
Close: 8451.19043
Volume: 674920000
Adj_Close: null
daily_range: 12.924443321266926
News: null
Explanation_report: null

Date: '2008-11-13'
Open: 8281.139648
High: 8876.589844
Low: 7965.419922
Close: 8835.25
Volume: 476600000
Adj_Close: null
daily_range: 11.439069514507388
News: null
Explanation_report: null

Date: '2008-10-13'
Open: 8462.419922
High: 9427.990234
Low: 8462.179688
Close: 9387.610352
Volume: 399290000
Adj_Close: null
daily_range: 11.413259722782675
News: null
Explanation_report: null

Date: '2008-10-28'
Open: 8178.720215
High: 9082.080078
Low: 8174.72998
Close: 9065.120117
Volume: 372160000
Adj_Close: null
daily_range: 11.099450382090795
News: null
Explanation_report: null

Date: '2010-05-06'
Open: 10868.120117
High: 10879.759766
Low: 9869.620117
Close: 10520.320312
Volume: 459890000
Adj_Close: null
daily_range: 10.23483819058118
News: null
Explanation_repo

## Complex AMAPs

aMaps function can contain external API and LLM calls. This way we can use agentics as a scaleout frameworks for complex workflows.

In [8]:
from ddgs import DDGS

## Define a function to get news for a given date using the DDGS search engine
## Note that the similar functionalities can be implemented using MCP tools in AGs
async def get_news(state):
    state.News=str(DDGS().text(f"What happended to the stock market and dow jones on {state.Date}", max_results=10))
    return state

## set the batch size for the amap function to 5 (only 10 states will be processed)
highest_volatility_days.batch_size = 10

# Now get news for the top 10 days with highest volatility using amap
highest_volatility_days = await highest_volatility_days.amap(get_news)

# print the first result for brevity
print(f"Date: {highest_volatility_days[0].Date}, Daily Range: {highest_volatility_days[0].daily_range}, News: {highest_volatility_days[0].News[:200]}...")

2025-09-03 19:30:47.685 | DEBUG    | agentics.core.agentics:amap:206 - Executing amap on function <function get_news at 0x7992c5befec0>
2025-09-03 19:30:55.488 | DEBUG    | agentics.core.agentics:amap:231 - 10 states processed. 0.7802352428436279 seconds average per state in the last chunk ...


Date: 2008-10-10, Daily Range: 12.924443321266926, News: [{'title': 'Dow Jones Industrial Average - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', 'body': 'The Dow Jones Industrial Average daily closing value plotted on a ...


Now let's use self transduction to provide an explanation for the market volatility

In [9]:
from agentics.core.llm_connections import get_llm_provider
highest_volatility_days.instructions = """Explain the reasons why the market went down or up
given the high volatility in the stock market on this day based on the news provided.
Provide a concise summary."""
highest_volatility_days.llm= get_llm_provider() ## You can choose between "openai", "watsonx", "gemini", "vllm_crewai"
highest_volatility_explanations = await highest_volatility_days.self_transduction(
["Date", "Open", "High", "Low", "Close", "Volume", "daily_range", "News"],["Explanation_report"])

for state in highest_volatility_explanations:
    print(f"Date: {state.Date}, Daily Range: {state.daily_range}\nExplanation: {state.Explanation_report}...")

2025-09-03 19:31:12.889 | DEBUG    | agentics.core.llm_connections:get_llm_provider:29 - No LLM provider specified. Using the first available provider.
2025-09-03 19:31:12.891 | DEBUG    | agentics.core.llm_connections:get_llm_provider:31 - Available LLM providers: ['watsonx']. Using 'watsonx'
2025-09-03 19:31:12.894 | DEBUG    | agentics.core.agentics:__lshift__:518 - Executing task: Explain the reasons why the market went down or up
given the high volatility in the stock market on this day based on the news provided.
Provide a concise summary.
10 states will be transduced
2025-09-03 19:31:12.896 | DEBUG    | agentics.core.agentics:__lshift__:612 - transducer class: <class 'agentics.abstractions.pydantic_transducer.PydanticTransducerCrewAI'>
2025-09-03 19:31:25.679 | DEBUG    | agentics.core.agentics:__lshift__:648 - Processed 10 states in 12.781238317489624 seconds
2025-09-03 19:31:25.680 | DEBUG    | agentics.core.agentics:__lshift__:700 - 10 states processed in 1.2781238317489625 s

Date: 2008-10-10, Daily Range: 12.924443321266926
Explanation: The market went down on this day due to high volatility in the stock market. The Dow Jones Industrial Average daily closing value was affected by various news and events, including the 2008 stock market return, which was more than 10% per year on average. Additionally, the Dow Jones Index experienced a significant decline, with a loss of about 1,600 points at one point, which is more than twice as large as the biggest Dow point decline ever. The stock market was also impacted by President Donald Trump's tariff plan, which led to major losses in both the Dow Jones and NASDAQ. Historically, the month of September is typically the worst month of the year for the stock market, with the Dow Jones Industrial Average generating an average monthly decline of 1.1% and finishing higher only 42.2% of the time dating back to 1897. However, on the day in question, the Dow Jones Industrial Average opened at 8568.669922, reached a high of

## Well Done
You are now fully equipped to work with agentics and apply it to your data.
Congratulations and please contribute back to the community if you feel this is exciting.